# Probabilistic robotics lab: tracking a moving vehicle
**Name:** Solution  
**Version:** 1.0 — 25 October 2026


In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
plt.rcParams["figure.dpi"] = 110
print("Imports OK:", "NumPy", np.__version__, "| pandas", pd.__version__)

Imports OK: NumPy 2.5.4 | pandas 3.1.0


In [18]:
rng = np.random.default_rng(SEED)
dt = 1.0
n_steps = 120
time = np.arange(n_steps) * dt

acceleration = np.zeros(n_steps)
acceleration[30:40] = 0.18
acceleration[75:85] = -0.12

truth = np.zeros((n_steps, 2))
truth[0] = [10.0, 1.0]
for t in range(1, n_steps):
    p, v = truth[t - 1]
    a = acceleration[t]
    truth[t, 0] = p + v * dt + 0.5 * a * dt**2 + rng.normal(0, 0.2)
    truth[t, 1] = v + a * dt + rng.normal(0, 0.1)

sensor_sd = 2.0
z_position = truth[:, 0] + rng.normal(0, sensor_sd, n_steps)
print("Data ready.")

Data ready.


In [19]:
def predict(mu, P, F, Q):
    mu_pred = F @ mu
    P_pred = F @ P @ F.T + Q
    return mu_pred, P_pred

def correct_position(mu_pred, P_pred, z, H, R):
    nu = z - H @ mu_pred
    S = H @ P_pred @ H + R
    K = (P_pred @ H) / S
    mu = mu_pred + K * nu
    P = P_pred - np.outer(K, H @ P_pred)
    return mu, P

def run_kf(z, F, H, Q, R, mu0, P0):
    n = len(z)
    means = np.zeros((n, 2))
    covariances = np.zeros((n, 2, 2))
    means[0] = mu0.copy()
    covariances[0] = P0.copy()
    for t in range(1, n):
        mu_pred, P_pred = predict(means[t - 1], covariances[t - 1], F, Q)
        if np.isnan(z[t]):
            means[t] = mu_pred
            covariances[t] = P_pred
        else:
            means[t], covariances[t] = correct_position(mu_pred, P_pred, z[t], H, R)
    return means, covariances

In [20]:
mu0 = np.array([8.0, 0.0])
P0 = np.diag([9.0, 1.0])
Q_base = np.diag([0.04, 0.01])
R_base = sensor_sd**2
F = np.array([[1.0, dt], [0.0, 1.0]])
H = np.array([1.0, 0.0])

def rmse(estimate, actual):
    return np.sqrt(np.mean((estimate - actual)**2))

kf_means, kf_covariances = run_kf(z_position, F, H, Q_base, R_base, mu0, P0)

pos_rmse_raw = rmse(z_position[1:], truth[1:, 0])
pos_rmse_kf = rmse(kf_means[1:, 0], truth[1:, 0])
vel_rmse_kf = rmse(kf_means[1:, 1], truth[1:, 1])

print(f"Raw Position RMSE: {pos_rmse_raw:.8f}")
print(f"KF Position RMSE: {pos_rmse_kf:.8f}")
print(f"KF Velocity RMSE: {vel_rmse_kf:.8f}")

Raw Position RMSE: 2.05352179
KF Position RMSE: 1.35605928
KF Velocity RMSE: 0.43682121


### Your answer — Task 1

**1.** Raw position RMSE is approximately 2.05 m, whereas the KF position RMSE is 1.36 m (1.356059 m), and the KF velocity RMSE is 0.44 m/s. Yes, filtering improved position accuracy, reducing the position error by over 33% relative to raw measurements.

**2.** Position measurements update velocity through off-diagonal covariance terms ($P_{pv}$) built by the prediction model ($F$). When a position innovation $\nu_t$ occurs, the Kalman gain entry $K_v = P_{pv} / S_t$ applies a fraction of that innovation directly to adjust the velocity estimate $\mu_v$.

**3.** The velocity estimate performs poorly during the acceleration periods ($t = 30\text{--}40\text{ s}$ and $t = 75\text{--}85\text{ s}$). Because the filter assumes constant velocity ($a = 0$), it lags behind the true state while the vehicle is actively accelerating.